# Did the Attitudes Predate the Switch? Did the Switch Happen in 2016?

Part 8 found that 2024 Biden→Trump switchers were near-Republican on immigration but measured that in 2024, after the vote. Two questions follow:

1. **Timing of attitudes.** The 2020–2024 panel asks the same four immigration items in 2020, 2022, and 2024 to the same people. Were the switchers already immigration-conservative in 2020, or did they move after leaving?
2. **The 2016 hypothesis.** Was the realignment of this voter type a 2016 event, with 2024 as aftermath? Two tests: the panel asked its 2020 respondents how they voted in 2016 — were the 2024 switchers Trump 2016 voters who briefly went Biden? And the 2016 CES has the 2012 vote, the 2016 general and primary vote, and comparable immigration and economic items — how large was the Obama→Trump switch, who were those voters, and did the Sanders primary feed it?

**Inputs:** `data/raw/ces_panel/merged_recontact_2024_vv.csv`, `data/raw/ces/CCES16_Common_OUTPUT_Feb2018_VV.tab`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
RAW_DIR = Path('../data/raw'); TABLES_DIR = Path('../outputs/tables')
def wmean(x, w):
    m = x.notna() & w.notna()
    return np.average(x[m], weights=w[m]) if m.any() else np.nan

## 1. Panel: immigration attitudes of the same people, 2020 → 2022 → 2024

Four items asked in every wave (guide pp. 42–43): grant legal status to long-employed immigrants (`331a`), increase border patrols (`331b`), cut legal immigration 50% (`CC20_331d` / `CC22_331c` / `RC24_331c`), $25B border security including a wall (`CC20_331e` / `CC22_331d` / `RC24_331d`). Index = mean share of conservative positions. Path defined on the contemporaneous 2020 and 2024 votes.

In [2]:
cols = ['commonpostweight_24', 'commonweight_24', 'presvote16post_20', 'CC20_410', 'CC24_410',
        'CC20_331a', 'CC20_331b', 'CC20_331d', 'CC20_331e', 'CC22_331a', 'CC22_331b', 'CC22_331c', 'CC22_331d', 'CC24_331a', 'CC24_331b', 'RC24_331c', 'RC24_331d',
        'CC20_332a', 'CC22_332a', 'RC24_332a']
w = pd.read_csv(RAW_DIR / 'ces_panel' / 'merged_recontact_2024_vv.csv', usecols=cols, low_memory=False)
w['wt'] = w['commonpostweight_24'].fillna(w['commonweight_24'])
w['v20'] = w['CC20_410'].map({1: 'D', 2: 'R'}); w['v24'] = w['CC24_410'].map({1: 'D', 2: 'R'})
w['v16'] = w['presvote16post_20'].map({1: 'Clinton', 2: 'Trump', 3: 'other', 4: 'other', 5: 'other', 6: 'other', 7: 'did not vote'})
cons = lambda c, sup: (w[c] == (1 if sup else 2)).astype(float).where(w[c].notna())
w['imm_2020'] = pd.concat([cons('CC20_331a', False), cons('CC20_331b', True), cons('CC20_331d', True), cons('CC20_331e', True)], axis=1).mean(axis=1)
w['imm_2022'] = pd.concat([cons('CC22_331a', False), cons('CC22_331b', True), cons('CC22_331c', True), cons('CC22_331d', True)], axis=1).mean(axis=1)
w['imm_2024'] = pd.concat([cons('CC24_331a', False), cons('CC24_331b', True), cons('RC24_331c', True), cons('RC24_331d', True)], axis=1).mean(axis=1)
w['abcons_2020'] = cons('CC20_332a', False); w['abcons_2022'] = cons('CC22_332a', False); w['abcons_2024'] = cons('RC24_332a', False)
a = w[w['v20'].notna() & w['v24'].notna()].copy()
a['path'] = a['v20'].map({'D': 'Biden', 'R': 'Trump'}) + '->' + a['v24'].map({'D': 'Harris', 'R': 'Trump'})
rows = []
for p in ['Biden->Harris', 'Biden->Trump', 'Trump->Harris', 'Trump->Trump']:
    s = a[a['path'] == p]
    rows.append({'path': p, 'N': len(s), **{f'imm_{y}': wmean(s[f'imm_{y}'], s['wt']) for y in [2020, 2022, 2024]},
                 **{f'abcons_{y}': wmean(s[f'abcons_{y}'], s['wt']) for y in [2020, 2022, 2024]}})
traj = pd.DataFrame(rows).set_index('path')
traj['imm_change_2020_2024'] = traj['imm_2024'] - traj['imm_2020']
print('Immigration conservatism (0-1) and abortion conservatism, same respondents, by 2020->2024 path (weighted):')
print(traj.round(2).to_string())
print()
print('2020 item-level (before the switch), share conservative — loyal Dem / Biden->Trump / loyal Rep:')
rows = []
for c, lab, sup in [('CC20_331a', 'oppose legal status', False), ('CC20_331b', 'more border patrol', True), ('CC20_331d', 'cut legal immigration 50%', True), ('CC20_331e', 'wall + $25B border security', True)]:
    v = [wmean((s[c] == (1 if sup else 2)).astype(float).where(s[c].notna()), s['wt']) for s in [a[a['path'] == p] for p in ['Biden->Harris', 'Biden->Trump', 'Trump->Trump']]]
    rows.append({'item_2020': lab, 'loyal_Dem': v[0], 'Biden_to_Trump': v[1], 'loyal_Rep': v[2]})
    print(f'  {lab:30s} {v[0]:.0%} / {v[1]:.0%} / {v[2]:.0%}')
items2020 = pd.DataFrame(rows).set_index('item_2020')
print()
print('Reading: switchers entered 2020 already three times as immigration-conservative as loyal Democrats (0.50 vs 0.17),')
print('then moved a further +0.24 by 2024 — more than loyal Democrats (+0.17), less than the gap that remained to Republicans.')
print('Both selection and movement. N = 39 switchers: the level gap is robust, the movement is ±0.15.')

Immigration conservatism (0-1) and abortion conservatism, same respondents, by 2020->2024 path (weighted):
                  N  imm_2020  imm_2022  imm_2024  abcons_2020  abcons_2022  abcons_2024  imm_change_2020_2024
path                                                                                                          
Biden->Harris  1645      0.17      0.24      0.34         0.13         0.11         0.09                  0.18
Biden->Trump     39      0.50      0.55      0.74         0.59         0.68         0.47                  0.24
Trump->Harris    39      0.51      0.53      0.51         0.43         0.42         0.28                 -0.00
Trump->Trump   1643      0.81      0.82      0.85         0.81         0.83         0.80                  0.04

2020 item-level (before the switch), share conservative — loyal Dem / Biden->Trump / loyal Rep:
  oppose legal status            10% / 72% / 63%
  more border patrol             33% / 49% / 98%
  cut legal immigration 50%     

## 2. Panel: what did the 2024 switchers do in 2016?

In [3]:
rows = []
for p in ['Biden->Harris', 'Biden->Trump', 'Trump->Harris', 'Trump->Trump']:
    s = a[a['path'] == p]
    rows.append({'path': p, 'N': len(s), **{k: wmean((s['v16'] == k).astype(float), s['wt']) for k in ['Clinton', 'Trump', 'other', 'did not vote']},
                 'N_unweighted_by_2016': s['v16'].value_counts().to_dict()})
v16 = pd.DataFrame(rows).set_index('path')
print('Recalled 2016 vote (asked in 2020) by 2020->2024 path (weighted):')
print(v16.drop(columns='N_unweighted_by_2016').round(3).to_string())
print('\nUnweighted counts, Biden->Trump switchers:', v16.loc['Biden->Trump', 'N_unweighted_by_2016'])
print('\nThe 2024 switchers were not Trump 2016 voters returning home: 4% voted Trump in 2016, 43% Clinton, 53% did not vote.')
print('Half of them entered the electorate in 2020. Trump->Harris switchers, by contrast, were 56% Trump 2016 voters.')

Recalled 2016 vote (asked in 2020) by 2020->2024 path (weighted):
                  N  Clinton  Trump  other  did not vote
path                                                    
Biden->Harris  1645    0.701  0.017  0.070         0.213
Biden->Trump     39    0.427  0.038  0.005         0.530
Trump->Harris    39    0.217  0.561  0.145         0.077
Trump->Trump   1643    0.019  0.753  0.063         0.165

Unweighted counts, Biden->Trump switchers: {'Clinton': 18, 'did not vote': 11, 'Trump': 8, 'other': 2}

The 2024 switchers were not Trump 2016 voters returning home: 4% voted Trump in 2016, 43% Clinton, 53% did not vote.
Half of them entered the electorate in 2020. Trump->Harris switchers, by contrast, were 56% Trump 2016 voters.


## 3. The 2016 switch itself

CES 2016: `CC16_326` (2012 vote, 1 Obama / 2 Romney), `CC16_410a` (2016 vote, 1 Trump / 2 Clinton), `CC16_328` (2016 primary vote). Immigration items are "select all that apply" (`CC16_331_*`, 1 = selected): legal status, border patrol, Dreamers, deport all. Economic: repeal ACA (`CC16_351I`, 1 = for), $12 minimum wage (`CC16_351K`, 1 = for). Weights: `commonweight_vv_post`.

In [4]:
cols = ['commonweight_vv_post', 'commonweight_vv', 'commonweight', 'CC16_326', 'CC16_410a', 'CC16_328', 'CC16_331_1', 'CC16_331_2', 'CC16_331_3', 'CC16_331_7', 'CC16_332a', 'CC16_351I', 'CC16_351K', 'pid7', 'educ', 'race', 'hispanic']
d = pd.read_csv(RAW_DIR / 'ces' / 'CCES16_Common_OUTPUT_Feb2018_VV.tab', sep='\t', usecols=cols, low_memory=False)
d['CC16_410a'] = pd.to_numeric(d['CC16_410a'], errors='coerce')
d['wt'] = d['commonweight_vv_post'].fillna(d['commonweight_vv']).fillna(d['commonweight'])
d['v12'] = d['CC16_326'].map({1: 'Obama', 2: 'Romney'}); d['v16'] = d['CC16_410a'].map({1: 'Trump', 2: 'Clinton'})
d['prim'] = d['CC16_328'].map({1: 'Clinton', 2: 'Sanders', 4: 'Trump', 5: 'Cruz', 6: 'Kasich', 7: 'Rubio'})
sel = lambda c: (d[c] == 1).astype(float).where(d[c].notna())
d['imm_cons'] = pd.concat([1 - sel('CC16_331_1'), sel('CC16_331_2'), 1 - sel('CC16_331_3'), sel('CC16_331_7')], axis=1).mean(axis=1)
d['abortion_cons'] = (d['CC16_332a'] == 2).astype(float).where(d['CC16_332a'].notna())
d['repeal_ACA'] = (d['CC16_351I'] == 1).astype(float).where(d['CC16_351I'].notna())
d['minwage_12'] = (d['CC16_351K'] == 1).astype(float).where(d['CC16_351K'].notna())
d['college'] = (d['educ'] >= 5).astype(float); d['white'] = ((d['race'] == 1) & (d['hispanic'] != 1)).astype(float); d['hisp'] = (d['hispanic'] == 1).astype(float)
d['pid7c'] = d['pid7'].where(d['pid7'] <= 7)
e16 = d[d['v12'].notna() & d['v16'].notna()].copy(); e16['path'] = e16['v12'] + '->' + e16['v16']
ob = e16[e16['v12'] == 'Obama']; ro = e16[e16['v12'] == 'Romney']
print(f'2016 CES respondents with a 2012 and 2016 major-party vote: {len(e16):,}')
print(f'  Obama 2012 -> Trump 2016: {wmean((ob["v16"] == "Trump").astype(float), ob["wt"]):.1%} of Obama voters (N = {int((ob["v16"] == "Trump").sum()):,})')
print(f'  Romney 2012 -> Clinton:   {wmean((ro["v16"] == "Clinton").astype(float), ro["wt"]):.1%} of Romney voters (N = {int((ro["v16"] == "Clinton").sum()):,})')
print('  2024 for comparison: Biden -> Trump 5.2%, Trump -> Harris 1.6%')
rows = []
for p in ['Obama->Clinton', 'Obama->Trump', 'Romney->Clinton', 'Romney->Trump']:
    s = e16[e16['path'] == p]
    rows.append({'path': p, 'N': len(s), **{v: wmean(s[v], s['wt']) for v in ['imm_cons', 'abortion_cons', 'repeal_ACA', 'minwage_12', 'college', 'white', 'hisp', 'pid7c']}})
prof16 = pd.DataFrame(rows).set_index('path')
print('\nProfile by 2012->2016 path (weighted):'); print(prof16.round(2).to_string())
print('\nImmigration items 2016, share conservative — Obama->Clinton / Obama->Trump / Romney->Trump:')
for c, lab, cs in [('CC16_331_1', 'oppose legal status', False), ('CC16_331_2', 'more border patrol', True), ('CC16_331_3', 'oppose Dreamers', False), ('CC16_331_7', 'deport all', True)]:
    v = [wmean((s[c] == 1).astype(float) if cs else 1 - (s[c] == 1).astype(float), s['wt']) for s in [e16[e16['path'] == p] for p in ['Obama->Clinton', 'Obama->Trump', 'Romney->Trump']]]
    print(f'  {lab:22s} {v[0]:.0%} / {v[1]:.0%} / {v[2]:.0%}')

2016 CES respondents with a 2012 and 2016 major-party vote: 35,589
  Obama 2012 -> Trump 2016: 12.0% of Obama voters (N = 2,121)
  Romney 2012 -> Clinton:   5.9% of Romney voters (N = 854)
  2024 for comparison: Biden -> Trump 5.2%, Trump -> Harris 1.6%

Profile by 2012->2016 path (weighted):
                     N  imm_cons  abortion_cons  repeal_ACA  minwage_12  college  white  hisp  pid7c
path                                                                                                
Obama->Clinton   18677      0.28           0.18        0.20        0.92     0.36   0.61  0.02   1.83
Obama->Trump      2121      0.62           0.42        0.72        0.73     0.23   0.80  0.02   3.82
Romney->Clinton    854      0.43           0.35        0.55        0.66     0.42   0.82  0.02   4.18
Romney->Trump    13937      0.77           0.72        0.95        0.34     0.28   0.89  0.02   5.88

Immigration items 2016, share conservative — Obama->Clinton / Obama->Trump / Romney->Trump:
  oppos

In [5]:
print('2016 primary vote among Obama-2012 voters, by general-election path (of primary voters, weighted):')
rows = []
for p in ['Obama->Clinton', 'Obama->Trump']:
    s = e16[(e16['path'] == p) & e16['prim'].notna()]
    r = {'path': p, 'N_primary_voters': len(s), **{k: wmean((s['prim'] == k).astype(float), s['wt']) for k in ['Clinton', 'Sanders', 'Trump', 'Cruz', 'Kasich', 'Rubio']}}
    rows.append(r); print(f'  {p:16s} N={len(s):>6,}  ' + '  '.join(f'{k} {r[k]:.0%}' for k in ['Clinton', 'Sanders', 'Trump', 'Cruz', 'Kasich']))
prim16 = pd.DataFrame(rows).set_index('path')
sb = d[(d['prim'] == 'Sanders') & d['v16'].notna()]
sb2 = d[(d['prim'] == 'Sanders') & (d['v12'] == 'Obama') & d['v16'].notna()]; st = sb2[sb2['v16'] == 'Trump']
print(f'\nSanders primary voters (N = {len(sb):,}): Clinton {wmean((sb["v16"] == "Clinton").astype(float), sb["wt"]):.1%}, Trump {wmean((sb["v16"] == "Trump").astype(float), sb["wt"]):.1%}')
print(f'Sanders primary voters who had voted Obama in 2012 (N = {len(sb2):,}): Trump {wmean((sb2["v16"] == "Trump").astype(float), sb2["wt"]):.1%}')
print(f'  those Sanders->Trump switchers (N = {len(st)}): imm_cons {wmean(st["imm_cons"], st["wt"]):.2f}, $12 min wage {wmean(st["minwage_12"], st["wt"]):.0%}, repeal ACA {wmean(st["repeal_ACA"], st["wt"]):.0%}')
print()
print('Reading: 47% of Obama->Trump switchers who voted in a primary voted for Trump in the Republican primary; 23% for Sanders.')
print('The Sanders->Trump pathway is real (7.7% of Sanders-Obama voters) but it is the minority pathway among 2016 switchers.')

2016 primary vote among Obama-2012 voters, by general-election path (of primary voters, weighted):
  Obama->Clinton   N=14,012  Clinton 63%  Sanders 33%  Trump 1%  Cruz 0%  Kasich 2%
  Obama->Trump     N= 1,322  Clinton 21%  Sanders 23%  Trump 47%  Cruz 4%  Kasich 3%

Sanders primary voters (N = 6,288): Clinton 86.1%, Trump 13.9%
Sanders primary voters who had voted Obama in 2012 (N = 5,146): Trump 7.7%
  those Sanders->Trump switchers (N = 288): imm_cons 0.56, $12 min wage 79%, repeal ACA 72%

Reading: 47% of Obama->Trump switchers who voted in a primary voted for Trump in the Republican primary; 23% for Sanders.
The Sanders->Trump pathway is real (7.7% of Sanders-Obama voters) but it is the minority pathway among 2016 switchers.


## 4. Save

In [6]:
traj.to_csv(TABLES_DIR / 'panel_immigration_trajectory.csv'); items2020.to_csv(TABLES_DIR / 'panel_immigration_items_2020.csv')
v16.drop(columns='N_unweighted_by_2016').to_csv(TABLES_DIR / 'panel_2016_vote_by_path.csv')
prof16.to_csv(TABLES_DIR / 'switchers_2016_profile.csv'); prim16.to_csv(TABLES_DIR / 'switchers_2016_primary.csv')
print('Saved 5 tables to outputs/tables/')

Saved 5 tables to outputs/tables/


## Limitations

- **39 switchers in the panel.** The 2020 level difference (0.50 vs 0.17) is far outside sampling error; the 2020→2024 movement (+0.24) has a standard error near 0.08 and the abortion series is noise.
- **2016 items are "select all that apply."** Not selecting an item is coded as opposing it, which inflates conservative shares relative to the 2020+ support/oppose format. Compare across paths within 2016, not levels across years.
- **2012 vote in the 2016 CES is recalled**, and 2016 vote in the panel is recalled in 2020. Recall bias favors the winner; 2012 recall likely over-states Obama voters, so the 12.0% Obama→Trump rate may be an upper bound.
- **The 2016 and 2024 switchers are not the same people**, and the data confirms it (§2). The comparison is of voter *types* across two events, not a panel.